# Recruitment Process Optimisation — People Analytics Case Study

**Author:** Denerson  
**Objective:** Analyse one year of recruitment data to help the Talent Acquisition
team understand process efficiency, identify funnel bottlenecks, and evaluate the
effectiveness of different sourcing channels.

This notebook is structured to mirror the final presentation, so it reads
top-to-bottom as a single narrative.

## Business Questions

1. **Sourcing channel effectiveness** - which channels perform best on *volume* of
   hires and *efficiency* (conversion rate)?
2. **Time to fill** - average time from requisition opened to hire, and how it
   varies by department.
3. **Funnel health** - conversion rates between hiring stages and where the biggest
   drop-offs are.
4. **Recommendations** - top 2–3 actionable improvements for the team.


## Notebook Structure

- **0. Setup & Data Loading** — imports, load raw data, first look.
- **1. Data Preparation** — parse dates, handle data quality issues, reconstruct
  the funnel.
- **2. Sourcing Channels** — volume vs. conversion efficiency.
- **3. Funnel Health** — stage-to-stage conversion and drop-offs.
- **4. Time to Fill** — by department (mean and median).
- **5. Synthesis & Recommendations** — turning findings into actions.


## 0. Setup & Data Loading

In [1]:
import pandas as pd

In [7]:
df = pd.read_csv('data/raw/applications_data.csv')
df.shape

(1049, 12)

In [8]:
import os
print(os.listdir('.'))

['.DS_Store', 'requirements.txt', 'recruitment_funnel_analsis.ipynb', 'docs', '.gitignore', '.ipynb_checkpoints', 'venv', '.git', 'data', 'outputs', 'notebooks']


In [9]:
print(os.listdir('data/raw/'))

['.gitkeep', 'applications_data.csv']


## 1. Data Preparation

### 1.1 Investigating date coherence
Requisitions open in 2024–2025, but application/hire dates extend into 2026–2027.
Before computing time to fill, we check whether this is a data quality issue.


In [13]:
# Parse date columns explicitly (m/d/Y) so downstream date arithmetic works.
date_cols = ['application_date', 'date_offer_extended',
             'date_hired', 'date_requisition_opened']

for col in date_cols:
    df[col] = pd.to_datetime(df[col], format='%m/%d/%Y', errors='coerce')

df[date_cols].dtypes



application_date           datetime64[ns]
date_offer_extended        datetime64[ns]
date_hired                 datetime64[ns]
date_requisition_opened    datetime64[ns]
dtype: object

In [14]:
# Requisitions open in 2024–2025 but some applications land in 2026–2027, which is
# suspicious. We measure the gap between a req opening and the application to see
# whether it's a plausible business pattern or a data quality problem.
df['days_req_to_app'] = (df['application_date'] - df['date_requisition_opened']).dt.days
df['days_req_to_app'].describe()

count    1049.000000
mean      341.768351
std       228.929888
min         7.000000
25%       154.000000
50%       297.000000
75%       502.000000
max       923.000000
Name: days_req_to_app, dtype: float64

In [15]:
# A 180-day threshold is a deliberate hight line: waiting half a year between a
# req opening and an application is hard to justify operationally, so anything
# beyond it is a candidate for closer inspection.
implausible = df[df['days_req_to_app'] > 180]
print("Rows with gap > 180 days:", len(implausible), f"({len(implausible)/len(df):.0%})")
print("\nMax gap (days):", df['days_req_to_app'].max())


Rows with gap > 180 days: 741 (71%)

Max gap (days): 923


In [16]:
# If the anomaly were real, we'd expect it to cluster somewhere (a slow department,
# a hard-to-fill role). If it's spread evenly, that points to systematic noise
# rather than a genuine signal.
print("By department:")
print(implausible['department'].value_counts(normalize=True).round(2))
print("\nOutcome breakdown:")
print(implausible['current_stage'].value_counts().head())



By department:
department
Sales         0.29
Technology    0.26
Marketing     0.11
Operations    0.11
Finance       0.11
HR            0.11
Name: proportion, dtype: float64

Outcome breakdown:
current_stage
Rejected                    430
Hired                       226
Hiring Manager Interview     25
Applied                      17
Technical Assessment         14
Name: count, dtype: int64


### Decision: time-to-fill metric

71% of applications arrive more than 180 days after the requisition opens
(max 923), and this is spread evenly across departments. This points to a data
artifact, not a real pattern. Filtering would remove most of the data, so we keep
the official metric (`requisition_opened → hired`) with a caveat, and we also add
`application → hired` as the measure the team actually controls and can act on.




In [18]:
# Official metric, kept as asked but skewed by the sourcing-gap artifact.
df['time_to_fill'] = (df['date_hired'] - df['date_requisition_opened']).dt.days

# The part TA actually controls: screening + interviews - the clean signal.
df['time_to_hire'] = (df['date_hired'] - df['application_date']).dt.days

df[['time_to_fill', 'time_to_hire']].describe()



,time_to_fill,time_to_hire
count,295.000000,295.000000
mean,428.508475,39.630508
std,237.091731,1.678926
min,61.000000,33.000000
25%,230.000000,39.000000
50%,412.000000,39.000000
75%,596.500000,40.000000
max,962.000000,48.000000
